In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
np.random.seed(0)

np.random.seed(0)
N = 100
T = 1000
input_amp = 0.01

def phi(Z):
    return Z / np.sqrt(1.0 + np.abs(Z)**2)

lap = np.zeros((N, N), dtype=np.float64)
lap[0, 0] = -10
lap[1, 0] = 1
lap[-1, 0] = 1
lap[0, 1] = 1
lap[0, -1] = 1

A_kernel = 1j * lap  # iΔ is anti-Hermitian
A_hat = np.fft.fft2(A_kernel)
U_hat = np.exp(A_hat)
print("max(|U_hat|-1):", np.max(np.abs(np.abs(U_hat) - 1.0)))

I = np.zeros((N, N), dtype=np.complex128)
I[N//2, N//2] = input_amp

Z = np.zeros((N, N), dtype=np.complex128)

def step(Z, U_hat, I):
    Z_lin = np.fft.ifft2(U_hat * np.fft.fft2(Z)) + I
    return phi(Z_lin)

for t in range(T):
    Z = step(Z, U_hat, I)

print("done; |Z| max =", np.max(np.abs(Z)))


from matplotlib.colors import hsv_to_rgb

mag = np.abs(Z)
scale = np.percentile(mag, 99) if np.any(mag) else 1.0
scale = float(scale) if scale > 0 else 1.0
v = np.clip(mag / scale, 0.0, 1.0)
h = (np.angle(Z) + np.pi) / (2 * np.pi)  # [-pi,pi] -> [0,1]
s = np.ones_like(h)
rgb = hsv_to_rgb(np.stack([h, s, v], axis=-1))

plt.figure(figsize=(5, 5))
plt.imshow(rgb, origin="lower", interpolation="nearest")
plt.title(f"|Z| brightness + phase hue (rainbow)\n(after {T} steps, input_amp={input_amp}, |Z| p99={scale:.3g})")
plt.axis("off")
plt.show()

In [ ]:
# GIF of Z (|Z| brightness + phase hue), timepoints 0 to 100
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import hsv_to_rgb
from IPython.display import Image, display

# --- Settings ---
t_start, t_end = 0, 1000  # inclusive
fps = 20
out_gif = Path("Z_t0_100.gif")

# --- Preconditions ---
required = ["N", "U_hat", "I", "step"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing variables from earlier cells: "
        + ", ".join(missing)
        + ". Run the setup/simulation cells above first."
    )

def complex_to_rgb(Z_frame: np.ndarray, scale: float) -> np.ndarray:
    mag = np.abs(Z_frame)
    scale = float(scale) if float(scale) > 0 else 1.0
    v = np.clip(mag / scale, 0.0, 1.0)
    h = (np.angle(Z_frame) + np.pi) / (2 * np.pi)  # [-pi,pi] -> [0,1]
    s = np.ones_like(h)
    return hsv_to_rgb(np.stack([h, s, v], axis=-1))

# --- Simulate frames from t_start..t_end (starting from Z=0) ---
Z_tmp = np.zeros((N, N), dtype=np.complex128)
Z_frames = [Z_tmp.copy()]  # t=0
for _t in range(t_end):
    Z_tmp = step(Z_tmp, U_hat, I)
    Z_frames.append(Z_tmp.copy())

# Global brightness scale for consistent visualization
all_mags = np.abs(np.stack(Z_frames, axis=0))
scale = np.percentile(all_mags, 99) if np.any(all_mags) else 1.0
scale = float(scale) if scale > 0 else 1.0

# --- Animate ---
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(complex_to_rgb(Z_frames[0], scale), origin="lower", interpolation="nearest")
ax.axis("off")
txt = ax.set_title(f"Z: |Z| brightness + phase hue (rainbow)\n(t={t_start}, |Z| p99={scale:.3g})")

def _update(i: int):
    im.set_data(complex_to_rgb(Z_frames[i], scale))
    txt.set_text(f"Z: |Z| brightness + phase hue (rainbow)\n(t={t_start + i}, |Z| p99={scale:.3g})")
    return (im, txt)

anim = FuncAnimation(fig, _update, frames=len(Z_frames), interval=1000 / fps, blit=False)

# --- Save GIF + display ---
anim.save(out_gif.as_posix(), writer=PillowWriter(fps=fps), dpi=150)
plt.close(fig)
display(Image(filename=out_gif.as_posix()))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from mpl_toolkits.mplot3d import Axes3D

# --- magnitude/phase → RGB ---
mag = np.abs(Z)
scale = np.percentile(mag, 99) if np.any(mag) else 1.0
scale = scale if scale > 0 else 1.0
v = np.clip(mag / scale, 0.0, 1.0)

h = (np.angle(Z) + np.pi) / (2*np.pi)
s = np.ones_like(h)

rgb = hsv_to_rgb(np.stack([h, s, v], axis=-1))

# --- torus geometry ---
N_torus = Z.shape[0]
theta_t = 2*np.pi * np.arange(N_torus) / N_torus
phi_t   = 2*np.pi * np.arange(N_torus) / N_torus
theta_t, phi_t = np.meshgrid(theta_t, phi_t, indexing="ij")

R = 2.5   # major radius
r = 1.0   # minor radius

X = (R + r*np.cos(phi_t)) * np.cos(theta_t)
Y = (R + r*np.cos(phi_t)) * np.sin(theta_t)
Z_torus = r * np.sin(phi_t)

# --- plot ---
fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(
    X, Y, Z_torus,
    facecolors=rgb,
    rstride=1, cstride=1,
    linewidth=0,
    antialiased=False,
    shade=False
)

ax.set_axis_off()
ax.set_box_aspect([1,1,1])
ax.set_title("Convolutional RNN activity on a flat torus\n(brightness = |Z|, hue = phase)")

plt.show()


In [ ]:
# GIF of torus visualization (|Z| brightness + phase hue), timepoints 0 to 100
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import hsv_to_rgb
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import Image, display

# --- Settings ---
t_start, t_end = 0, 10000  # inclusive
fps = 20
out_gif = Path("Z_torus_t0_100.gif")
frame_skip = 100

# --- Preconditions ---
required = ["N", "U_hat", "I", "step"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing variables from earlier cells: "
        + ", ".join(missing)
        + ". Run the setup/simulation cells above first."
    )

def complex_to_rgb_torus(Z_frame: np.ndarray, scale: float) -> np.ndarray:
    """Convert complex array to RGB using magnitude as brightness and phase as hue."""
    mag = np.abs(Z_frame)
    scale = float(scale) if float(scale) > 0 else 1.0
    v = np.clip(mag / scale, 0.0, 1.0)
    h = (np.angle(Z_frame) + np.pi) / (2 * np.pi)  # [-pi,pi] -> [0,1]
    s = np.ones_like(h)
    return hsv_to_rgb(np.stack([h, s, v], axis=-1))

# --- Simulate frames, skip some ---
Z_tmp = np.zeros((N, N), dtype=np.complex128)
Z_frames_torus = [Z_tmp.copy()]  # t=0
for _t in range(1, t_end + 1):
    Z_tmp = step(Z_tmp, U_hat, I)
    if _t % frame_skip == 0:
        Z_frames_torus.append(Z_tmp.copy())

# Global brightness scale for consistent visualization
all_mags = np.abs(np.stack(Z_frames_torus, axis=0))
scale = np.percentile(all_mags, 99) if np.any(all_mags) else 1.0
scale = float(scale) if scale > 0 else 1.0

# --- Torus geometry ---
N_grid = Z_frames_torus[0].shape[0]
theta_torus = 2 * np.pi * np.arange(N_grid) / N_grid
phi_torus = 2 * np.pi * np.arange(N_grid) / N_grid
theta_torus, phi_torus = np.meshgrid(theta_torus, phi_torus, indexing="ij")

R = 2.5  # major radius
r = 1.0  # minor radius

X_torus = (R + r * np.cos(phi_torus)) * np.cos(theta_torus)
Y_torus = (R + r * np.cos(phi_torus)) * np.sin(theta_torus)
Z_torus_geom = r * np.sin(phi_torus)

# --- Animate ---
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")

# Initial plot
rgb_init = complex_to_rgb_torus(Z_frames_torus[0], scale)
ax.plot_surface(
    X_torus, Y_torus, Z_torus_geom,
    facecolors=rgb_init,
    rstride=1, cstride=1,
    linewidth=0,
    antialiased=False,
    shade=False
)
ax.set_axis_off()
ax.set_box_aspect([1, 1, 1])
ax.set_title(f"Convolutional RNN activity on a flat torus\n(t={t_start}, |Z| p99={scale:.3g})")

def _update_torus(i: int):
    ax.clear()
    rgb = complex_to_rgb_torus(Z_frames_torus[i], scale)
    ax.plot_surface(
        X_torus, Y_torus, Z_torus_geom,
        facecolors=rgb,
        rstride=1, cstride=1,
        linewidth=0,
        antialiased=False,
        shade=False
    )
    ax.set_axis_off()
    ax.set_box_aspect([1, 1, 1])
    ax.set_title(f"Convolutional RNN activity on a flat torus\n(t={t_start + 100*i}, |Z| p99={scale:.3g})")

anim = FuncAnimation(fig, _update_torus, frames=len(Z_frames_torus), interval=1000 / fps, blit=False)

# --- Save GIF + display ---
print(f"Saving GIF to {out_gif.as_posix()}... ({len(Z_frames_torus)} frames)")
anim.save(out_gif.as_posix(), writer=PillowWriter(fps=fps), dpi=72)
plt.close(fig)
print("Done!")
display(Image(filename=out_gif.as_posix()))

In [ ]:
import networkx as nx
import numpy as np
from itertools import product
from scipy.optimize import linprog

# Generate random 4-regular graph
n = 30
G = nx.random_regular_graph(d=4, n=n, seed=0)

def ollivier_ricci(G, u, v):
    # neighbors including self excluded
    Nu = list(G.neighbors(u))
    Nv = list(G.neighbors(v))
    
    deg = len(Nu)
    mu = np.ones(deg) / deg
    mv = np.ones(deg) / deg
    
    # distance matrix between neighbors
    D = np.zeros((deg, deg))
    for i, x in enumerate(Nu):
        for j, y in enumerate(Nv):
            D[i, j] = nx.shortest_path_length(G, x, y)
    
    # solve Wasserstein distance via linear programming
    c = D.flatten()
    A_eq = []
    b_eq = []
    
    # row constraints
    for i in range(deg):
        row = np.zeros(deg*deg)
        row[i*deg:(i+1)*deg] = 1
        A_eq.append(row)
        b_eq.append(mu[i])
        
    # column constraints
    for j in range(deg):
        col = np.zeros(deg*deg)
        col[j::deg] = 1
        A_eq.append(col)
        b_eq.append(mv[j])
    
    bounds = [(0, None)] * (deg*deg)
    
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    W = res.fun
    
    return 1 - W

curvatures = []
for u, v in G.edges():
    curvatures.append(ollivier_ricci(G, u, v))

print("Average Ollivier–Ricci curvature:", np.mean(curvatures))